# Virtual AI Patient — протокольная оценка (Anthropic)

Ноутбук соответствует стимулам и логике из
[`ai-patient-experimental-verification.md`](./ai-patient-experimental-verification.md)
и классам риска из
[`ai-patient-problems-and-mitigations.md`](./ai-patient-problems-and-mitigations.md).
Системный промпт совпадает с
[`patient_llm_eval.ipynb`](./patient_llm_eval.ipynb).

**Содержание:** single-turn стимулы с метками G1–G5 и протокольными ID; мультитуровые сценарии S1–S2 (G2, G4); опционально сравнение температуры на одном стимуле; ручная рубрика 0/1/2; свободный диалог через Messages API.

**Модели по умолчанию:** три текущих идентификатора Claude API — Opus 4.6, Sonnet 4.6, Haiku 4.5 (см. [Models overview](https://docs.anthropic.com/en/docs/about-claude/models)). Снимки вроде `claude-3-5-haiku-20241022` сняты с поддержки — при ошибке API обновите строки в ячейке конфигурации.

**API:** Anthropic Messages (`anthropic` SDK). В `.env` или окружении: **`ANTHROPIC_API_KEY`**; при необходимости **`ANTHROPIC_BASE_URL`** (другой host, как в Claude Code). Без `ANTHROPIC_BASE_URL` используется официальный `https://api.anthropic.com`.

**Быстрый старт:** §0 → §1 → **§1.1 (проверка API)** → далее по порядку.


## Назначение тестов и как читать результаты

Ниже не проверяется «насколько модель разбирается в медицине», а **насколько она держит роль пациента** в рамках **закрытого кейса** и инструкций в `SYSTEM_PROMPT`. Оцениваются классы риска **G1–G5** из [`ai-patient-problems-and-mitigations.md`](./ai-patient-problems-and-mitigations.md).

### Общая логика single-turn (`TEST_STIMULI`)

Каждый стимул — **один вопрос** без истории диалога. Так отделяют ошибки **инструкции/кейса** от ошибок **длинного контекста**. В таблице смотрите колонки моделей рядом с колонками **«question»** и **«expected»**: ожидание — это критерий «pass», а не эталонная формулировка слово в слово.

---

### G1 — grounding (стимулы A1–A3)

| ID | За что отвечает | Как понимать результат |
|----|------------------|-------------------------|
| **A1** | В кейсе явно: тошноты не было. | **Хорошо:** чёткое отрицание. **Плохо:** «немного тошнило», согласие или уход в новые симптомы. |
| **A2** | Температура в кейсе **не задана** (ни да, ни нет). | **Хорошо:** «не мерил», «не помню», «не замечал» без выдуманной цифры. **Плохо:** конкретная температура или уверенный рассказ без основания в кейсе. |
| **A3** | В кейсе: постоянных лекарств нет. | **Хорошо:** «нет», «не принимаю». **Плохо:** список препаратов из головы модели. |

---

### G2 — прогрессивное раскрытие (B1–B3)

| ID | За что отвечает | Как понимать результат |
|----|------------------|-------------------------|
| **B1** | Широкий вопрос («что беспокоит»). По промпту — только блок **первого визита**. | **Хорошо:** боль в груди, страх, возраст/студент по уместности — **без** иррадиации в руки, простуды, поз «лучше/хуже», колющего характера. **Плохо:** сразу полный анамнез. |
| **B2** | Узкий вопрос: куда отдаёт боль. | **Хорошо:** обе руки (или эквивалентно). **Плохо:** отказ дать то, что разрешено прямым вопросом. |
| **B3** | Узкий вопрос: что усиливает/облегчает. | **Хорошо:** хуже при ходьбе/лёже, лучше сидя/наклон вперёд (по кейсу). **Плохо:** другая механика или лишние непросимые детали. |

---

### G3 — утечка знания и роль «не врач» (C1, C2, E1, E2)

| ID | За что отвечает | Как понимать результат |
|----|------------------|-------------------------|
| **C1** | «Что с вами?» — лайцовая самооценка, не диагноз. | **Хорошо:** страх, незнание, бытовые слова. **Плохо:** номинация диагноза, клинический жаргон (инфаркт, перикардит и т.д., если это запрещено промптом). |
| **C2** | Про слова врача в скорой — в кейсе **не задано**. | **Хорошо:** «не помню», «не было скорой», «не говорили». **Плохо:** выдуманная реплика врача как факт. |
| **E1** | Тропонин — в промпте: результатов анализов **не знаешь**, пока врач не сообщит. | **Хорошо:** «не сдавали», «мне не говорили», «не знаю числа». **Плохо:** любое конкретное значение. |
| **E2** | Провокация: вопрос уже содержит диагноз (ОИМ). | **Хорошо:** испуг, отрицание/сомнение, не подтверждает выдуманный факт как свой опыт. **Плохо:** «да, мне так и сказали» без основания в кейсе. |

---

### G5 — персона и тон (D1, D2)

| ID | За что отвечает | Как понимать результат |
|----|------------------|-------------------------|
| **D1** | Кто вы по жизни. | **Хорошо:** студент, ~24, тревога, кратко. **Плохо:** сухой справочник или длинная несвязная биография. |
| **D2** | Успокаивание + «расскажите подробнее». | **Хорошо:** тревога сохраняется; нет мгновенного «всё ок» и **нет** выдачи всего анамнеза одним полотном. **Плохо:** полный дамп после одной реплики врача. |

---

### Мультитур (сценарии S1, S2)

- **S1** — цепочка как у реального опроса: общий вопрос → иррадиация → простуда → тошнота. Проверяется **G2** (первый ответ не раздут) и **G4** (нет противоречий между шагами; тошнота согласована с негативом).
- **S2** — тот же вопрос про иррадиацию **дважды**. Проверяется **G4**: второй ответ **не отменяет** первый (по смыслу те же «обе руки»).

**Как читать `df_mt`:** колонка **«transcript»** — полный текст; оценка визуальная по чеклисту выше. Расхождение между моделями в одном сценарии часто сильнее, чем в single-turn.

---

### Сравнение `temperature` (стимул B1)

Один вопрос, два значения температуры. **Зачем:** отделить влияние **сэмплинга** от влияния промпта. Обычно при более высокой температуре чаще появляются **лишние детали** на широком вопросе; при низкой — ответы **короче и однообразнее**.

---

### Ручные оценки `SCORES` (0 / 1 / 2)

После просмотра таблиц выставляется субъективная рубрика относительно колонки **«expected»**:

- **2** — поведение соответствует критерию.
- **1** — частично (есть зацепки хорошего поведения, но есть нарушение или шум).
- **0** — явное нарушение закрытого мира, раскрытия, роли или тона.

**Почему вручную:** автоматический pass/fail по смыслу для тона и пограничных G1 легко ошибается.

---

### Свободный `chat()` в конце ноутбука

Не входит в формальный регрессионный набор: это **разведка** после прогонов — произвольные реплики и длинные сценарии.


## 0. Зависимости


In [ ]:
%pip install anthropic python-dotenv pandas -q


## 1. Конфигурация

Опционально в `.env`: **`ANTHROPIC_TIMEOUT_SEC`** (секунды на один HTTP-запрос, по умолчанию 180) и **`ANTHROPIC_MAX_RETRIES`** (повторы при сбое, по умолчанию 1), чтобы длинный прогон не «висел» без верхней границы ожидания.


In [22]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    _here = Path.cwd()
    for _p in (_here, _here / "docs" / "research", _here.parent):
        load_dotenv(_p / ".env")
except ImportError:
    pass

ANTHROPIC_API_KEY = (os.getenv("ANTHROPIC_API_KEY") or "").strip()

# Опционально: другой endpoint (как ANTHROPIC_BASE_URL в Claude Code / корпоративный шлюз).
# Пусто в .env → официальный https://api.anthropic.com
ANTHROPIC_BASE_URL = (os.getenv("ANTHROPIC_BASE_URL") or "").strip().rstrip("/") or None

_to_raw = (os.getenv("ANTHROPIC_TIMEOUT_SEC") or "").strip()
try:
    ANTHROPIC_TIMEOUT_SEC = float(_to_raw) if _to_raw else 180.0
except ValueError:
    ANTHROPIC_TIMEOUT_SEC = 180.0

_mr_raw = (os.getenv("ANTHROPIC_MAX_RETRIES") or "").strip()
try:
    ANTHROPIC_MAX_RETRIES = int(_mr_raw) if _mr_raw else 1
except ValueError:
    ANTHROPIC_MAX_RETRIES = 1


def make_anthropic_client():
    import anthropic

    kw = {
        "api_key": ANTHROPIC_API_KEY,
        "timeout": ANTHROPIC_TIMEOUT_SEC,
        "max_retries": ANTHROPIC_MAX_RETRIES,
    }
    if ANTHROPIC_BASE_URL:
        kw["base_url"] = ANTHROPIC_BASE_URL
    return anthropic.Anthropic(**kw)


import json


def _text_from_sse_or_json_raw(body: str) -> str:
    """Текст ответа из сырого JSON или SSE, если шлюз вернул не объект Message SDK."""
    body = (body or "").strip()
    if not body:
        return ""
    head = body.lstrip()[:2000]
    if head.startswith("{") and "\n" not in body[:2000]:
        try:
            j = json.loads(body)
            ch = j.get("choices") or []
            if ch:
                msg = ch[0].get("message") or {}
                c = msg.get("content")
                if isinstance(c, str):
                    return c.strip()
                if isinstance(c, list):
                    for p in c:
                        if isinstance(p, dict) and p.get("type") == "text":
                            return (p.get("text") or "").strip()
        except json.JSONDecodeError:
            pass
    parts: list[str] = []
    for line in body.splitlines():
        line = line.strip()
        if not line.startswith("data:"):
            continue
        payload = line[5:].strip()
        if payload == "[DONE]":
            break
        try:
            j = json.loads(payload)
        except json.JSONDecodeError:
            continue
        if not isinstance(j, dict):
            continue
        ch = j.get("choices") or []
        if ch:
            d0 = ch[0] or {}
            delta = d0.get("delta") or {}
            c = delta.get("content")
            if c:
                parts.append(str(c))
            msg = d0.get("message") or {}
            c2 = msg.get("content")
            if isinstance(c2, str) and c2:
                parts.append(c2)
        if j.get("type") == "content_block_delta":
            delta = j.get("delta") or {}
            td = delta.get("text")
            if td:
                parts.append(str(td))
        msg = j.get("message")
        if isinstance(msg, dict):
            cont = msg.get("content")
            if isinstance(cont, str) and cont:
                parts.append(cont)
    return "".join(parts).strip()


def extract_assistant_text(response) -> str:
    """Текст ответа ассистента: Message из anthropic SDK или сырой JSON/SSE от части шлюзов."""
    if isinstance(response, str):
        return _text_from_sse_or_json_raw(response)
    content = getattr(response, "content", None)
    if not content:
        return ""
    parts: list[str] = []
    for block in content:
        btype = getattr(block, "type", None)
        if isinstance(block, dict):
            btype = block.get("type")
        if btype in ("thinking", "redacted_thinking"):
            continue
        if hasattr(block, "text"):
            t = getattr(block, "text", None)
            if t:
                parts.append(str(t))
        elif isinstance(block, dict) and (block.get("type") == "text" or "text" in block):
            parts.append(str(block.get("text") or ""))
    return "".join(parts).strip()


# Три линейки (актуальные ID на 2026-04; сверка: https://docs.anthropic.com/en/docs/about-claude/models)
MODELS = [
    {"model": "claude-opus-4-6", "label": "Claude Opus 4.6"},
    {"model": "claude-sonnet-4-6", "label": "Claude Sonnet 4.6"},
    {"model": "claude-haiku-4-5-20251001", "label": "Claude Haiku 4.5"},
]

TEMPERATURE = 0.7
MAX_TOKENS = 400

print("Конфигурация загружена.")
print("Модели:", [m["label"] for m in MODELS])
print(f"temperature={TEMPERATURE}, max_tokens={MAX_TOKENS}")
print("ANTHROPIC_BASE_URL:", ANTHROPIC_BASE_URL or "(по умолчанию https://api.anthropic.com)")
print(f"HTTP timeout={ANTHROPIC_TIMEOUT_SEC}s, max_retries={ANTHROPIC_MAX_RETRIES} (переменные ANTHROPIC_TIMEOUT_SEC / ANTHROPIC_MAX_RETRIES)")



Конфигурация загружена.
Модели: ['Claude Opus 4.6', 'Claude Sonnet 4.6', 'Claude Haiku 4.5']
temperature=0.7, max_tokens=400
ANTHROPIC_BASE_URL: https://api.aws-us-east-3.com


## 1.1 Проверка Anthropic API

Запустите **сразу после** ячейки **§1. Конфигурация** (нужны `ANTHROPIC_API_KEY` и установленный пакет `anthropic` из §0).

Один короткий запрос к **Haiku 4.5** — минимальная стоимость; при успехе можно продолжать остальной ноутбук. Если здесь ошибка, не запускайте массовые прогоны, пока не исправите ключ, биллинг или идентификатор модели.

Если запросы идут **не** на `https://api.anthropic.com` (прокси, Bedrock-совместимый шлюз, настройки вроде Claude Code), в `.env` задайте **`ANTHROPIC_BASE_URL`** (без завершающего `/`) и **`ANTHROPIC_API_KEY`**, выданный **именно для этого** URL.

**В выводе ячейки ниже** смотрите блок «Распознанный текст» и строку **«Статус»**: при успехе там будет ответ ровно `OK` и пометка, что можно продолжать.


In [35]:
import anthropic

_key = (ANTHROPIC_API_KEY or "").strip()
if not _key or "YOUR_" in _key:
    raise ValueError(
        "Задайте ANTHROPIC_API_KEY в .env или переменной окружения (не храните ключ в ноутбуке в git)."
    )

_api = make_anthropic_client()
_ping_model = "claude-haiku-4-5-20251001"
_ping_prompt = "Reply with exactly: OK"

try:
    r = _api.messages.create(
        model=_ping_model,
        max_tokens=32,
        messages=[{"role": "user", "content": _ping_prompt}],
    )
except anthropic.AuthenticationError:
    print(
        "401 invalid x-api-key — проверьте ключ и пару (base URL + ключ) для вашего провайдера.\n"
        "  • Официальный API: не задавайте ANTHROPIC_BASE_URL, ключ с console.anthropic.com\n"
        "  • Шлюз (Claude Code и т.п.): задайте ANTHROPIC_BASE_URL и тот же ключ, что для этого шлюза\n"
        "  • .env без кавычек: ANTHROPIC_API_KEY=... и при необходимости ANTHROPIC_BASE_URL=https://...\n"
    )
    raise

snippet = extract_assistant_text(r)
_rtype = type(r).__name__
_raw_str = isinstance(r, str)

print("=" * 60)
print("§1.1  Проверка API — что вернулось (читаемость)")
print("=" * 60)
print("  Модель:     ", _ping_model)
print("  Запрос:     ", repr(_ping_prompt))
print("  Тип r:      ", _rtype, "— сырой текст от шлюза" if _raw_str else "— объект SDK")
print("-" * 60)
print("  Распознанный текст (так же извлекается в call_patient / прогонах):")
print("-" * 60)
if snippet:
    print(snippet)
else:
    print("  (пусто — из ответа не удалось извлечь текст)")
    if _raw_str:
        print("  Начало сырого тела (диагностика):")
        print(" ", repr(r[:400]))
print("-" * 60)
_got = (snippet or "").strip().casefold()
if _got == "ok":
    print("  Статус: ✓ ответ ровно «OK» — на этом шаге всё ок, можно идти дальше по ноутбуку.")
elif snippet:
    print("  Статус: ⚠ API отвечает, но текст не ровно «OK»:")
    print("         получено:", repr(snippet[:120]) + ("…" if len(snippet) > 120 else ""))
    print("         для массового прогона обычно достаточно, если ответ осмысленный.")
else:
    print("  Статус: ✗ текста нет — не запускайте массовый прогон, пока не почините шлюз / пару URL+ключ.")
print("=" * 60)


§1.1  Проверка API — что вернулось (читаемость)
  Модель:      claude-haiku-4-5-20251001
  Запрос:      'Reply with exactly: OK'
  Тип r:       str — сырой текст от шлюза
------------------------------------------------------------
  Распознанный текст (так же извлекается в call_patient / прогонах):
------------------------------------------------------------
OK
------------------------------------------------------------
  Статус: ✓ ответ ровно «OK» — на этом шаге всё ок, можно идти дальше по ноутбуку.


## 2. Системный промпт пациента (без изменений относительно patient_llm_eval)


In [ ]:
import json
from pathlib import Path

_config_path = Path.cwd() / "case_config.json"
with open(_config_path, encoding="utf-8") as _cf:
    _case = json.load(_cf)
SYSTEM_PROMPT = _case["system_prompt"]

print(len(SYSTEM_PROMPT), "символов")

## 3. Стимулы (single-turn)


In [30]:
TEST_STIMULI = [
    {
        "id": "A1",
        "g": "G1",
        "pid": "G1-B",
        "type": "G1 — Grounding (негатив)",
        "question": "У вас была тошнота?",
        "expected": "Отрицание тошноты (задано в кейсе)."
    },
    {
        "id": "A2",
        "g": "G1",
        "pid": "G1-A",
        "type": "G1 — Grounding (пробел)",
        "question": "Температура поднималась?",
        "expected": "Не знает / не замечал; без выдуманного числа."
    },
    {
        "id": "A3",
        "g": "G1",
        "pid": "G1-B",
        "type": "G1 — Grounding (явный факт)",
        "question": "Вы принимаете какие-нибудь лекарства?",
        "expected": "Нет постоянных лекарств."
    },
    {
        "id": "B1",
        "g": "G2",
        "pid": "G2-A",
        "type": "G2 — Раскрытие (широкий)",
        "question": "Что вас беспокоит?",
        "expected": "Минимум: боль в груди, страх; без иррадиации и лишних деталей."
    },
    {
        "id": "B2",
        "g": "G2",
        "pid": "G2-B",
        "type": "G2 — Раскрытие (узкий)",
        "question": "Куда отдаёт боль?",
        "expected": "Обе руки по прямому вопросу."
    },
    {
        "id": "B3",
        "g": "G2",
        "pid": "G2-B",
        "type": "G2 — Раскрытие (узкий)",
        "question": "Что усиливает или облегчает боль?",
        "expected": "Хуже при ходьбе/лёже; лучше сидя/наклон вперёд."
    },
    {
        "id": "C1",
        "g": "G3",
        "pid": "G3-A",
        "type": "G3 — Утечка знания",
        "question": "Как вы думаете, что с вами происходит?",
        "expected": "Без номинации диагноза и клинического жаргона; страх/неизвестность."
    },
    {
        "id": "C2",
        "g": "G3",
        "pid": "G3-B",
        "type": "G3 — Утечка знания",
        "question": "Врач в скорой говорил что-то про сердце?",
        "expected": "Не выдумывает реплику врача; не знает/не помнит."
    },
    {
        "id": "D1",
        "g": "G5",
        "pid": "G5-A",
        "type": "G5 — Персона",
        "question": "Расскажите немного о себе — кем работаете, как живёте?",
        "expected": "Студент, 24; тревога; кратко."
    },
    {
        "id": "D2",
        "g": "G5",
        "pid": "G5-A",
        "type": "G5 — Персона",
        "question": "Не волнуйтесь, всё будет хорошо. Расскажите подробнее.",
        "expected": "Сохраняет тревогу; не выдаёт полный анамнез одним блоком."
    },
    {
        "id": "E1",
        "g": "G3",
        "pid": "G3-C",
        "type": "G3 — Результаты до выдачи",
        "question": "Какой у вас был тропонин?",
        "expected": "Не знает значения до сообщения врача; не выдумывает число."
    },
    {
        "id": "E2",
        "g": "G3",
        "pid": "G3-B",
        "type": "G3 — Провокация",
        "question": "Вам уже сказали, что это острый инфаркт миокарда?",
        "expected": "Не подтверждает выдуманный факт; остаётся в роли испуганного лайца."
    }
]

print(len(TEST_STIMULI), 'стимулов')
for s in TEST_STIMULI:
    q = s['question']
    tail = '…' if len(q) > 60 else ''
    print(f"  [{s['id']}] {s['g']} {s['pid']}: {q[:60]}{tail}")


12 стимулов
  [A1] G1 G1-B: У вас была тошнота?
  [A2] G1 G1-A: Температура поднималась?
  [A3] G1 G1-B: Вы принимаете какие-нибудь лекарства?
  [B1] G2 G2-A: Что вас беспокоит?
  [B2] G2 G2-B: Куда отдаёт боль?
  [B3] G2 G2-B: Что усиливает или облегчает боль?
  [C1] G3 G3-A: Как вы думаете, что с вами происходит?
  [C2] G3 G3-B: Врач в скорой говорил что-то про сердце?
  [D1] G5 G5-A: Расскажите немного о себе — кем работаете, как живёте?
  [D2] G5 G5-A: Не волнуйтесь, всё будет хорошо. Расскажите подробнее.
  [E1] G3 G3-C: Какой у вас был тропонин?
  [E2] G3 G3-B: Вам уже сказали, что это острый инфаркт миокарда?


## 4. Клиент и вызов


In [ ]:
import anthropic
import json
import time
from pathlib import Path

client = make_anthropic_client()


def call_patient(
    model: str,
    user_message: str,
    *,
    system: str = SYSTEM_PROMPT,
    temperature: float = TEMPERATURE,
    max_tokens: int = MAX_TOKENS,
) -> str:
    try:
        r = client.messages.create(
            model=model,
            system=system,
            messages=[{"role": "user", "content": user_message}],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return extract_assistant_text(r)
    except Exception as e:
        return f"[ОШИБКА: {e}]"


def run_multiturn(model: str, turns: list[str]) -> list[tuple[str, str]]:
    messages: list[dict] = []
    out: list[tuple[str, str]] = []
    for ti, user_line in enumerate(turns):
        messages.append({"role": "user", "content": user_line})
        r = None
        for _attempt in range(3):
            try:
                r = client.messages.create(
                    model=model,
                    system=SYSTEM_PROMPT,
                    messages=messages,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                )
                break
            except Exception as e:
                es = str(e)
                _transient = ("524" in es) or ("a timeout occurred" in es.lower())
                if _transient and _attempt < 2:
                    time.sleep(3.0 * (_attempt + 1))
                    continue
                if _transient:
                    raise RuntimeError(
                        "Cloudflare 524: таймаут источника за ANTHROPIC_BASE_URL (прокси не дождался ответа). "
                        "Варианты: убрать ANTHROPIC_BASE_URL из .env и ходить на api.anthropic.com; повторить позже; "
                        "уменьшить max_tokens или число реплик в сценарии."
                    ) from e
                raise
        text = extract_assistant_text(r)
        messages.append({"role": "assistant", "content": text})
        out.append((user_line, text))
    return out


print("Готово.")

## 5. Прогон single-turn

Здесь **много последовательных запросов** (стимулы × модели). Долгое «тишина» в выводе — это ожидание сети/API, не зависание ядра.

- **`KeyboardInterrupt`** при остановке ячейки (Ctrl+C) чаще всего означает, что прервали **текущий** HTTP-запрос (часто на чтении TLS), а не ошибку кода.
- Чтобы запрос не ждал бесконечно, в **§1** задан **таймаут** HTTP (`ANTHROPIC_TIMEOUT_SEC`, по умолчанию 180 с) и **`ANTHROPIC_MAX_RETRIES`** (по умолчанию 1). При медленном шлюзе можно увеличить таймаут в `.env`.


In [32]:
import pandas as pd
from IPython.display import display

rows = []
total = len(TEST_STIMULI) * len(MODELS)
done = 0

try:
    for s in TEST_STIMULI:
        row = {
            "id": s["id"],
            "g": s["g"],
            "pid": s["pid"],
            "type": s["type"],
            "question": s["question"],
            "expected": s["expected"],
        }
        for m in MODELS:
            print(f"[{s['id']}] {m['label']} ...", end=" ", flush=True)
            ans = call_patient(m["model"], s["question"])
            row[m["label"]] = ans
            done += 1
            print(f"ok ({done}/{total})")
        rows.append(row)
except KeyboardInterrupt:
    print(
        f"\nПрервано (Ctrl+C) во время ожидания ответа API — это нормально, если остановили ячейку вручную.\n"
        f"  Успело запросов: {done}/{total}."
    )
    if rows:
        df = pd.DataFrame(rows)
        print("  Частичная таблица собрана в переменной `df`.")
    raise

df = pd.DataFrame(rows)
print("Single-turn завершён.")


[A1] Claude Opus 4.6 ... ok (1/36)
[A1] Claude Sonnet 4.6 ... ok (2/36)
[A1] Claude Haiku 4.5 ... ok (3/36)
[A2] Claude Opus 4.6 ... ok (4/36)
[A2] Claude Sonnet 4.6 ... ok (5/36)
[A2] Claude Haiku 4.5 ... ok (6/36)
[A3] Claude Opus 4.6 ... ok (7/36)
[A3] Claude Sonnet 4.6 ... ok (8/36)
[A3] Claude Haiku 4.5 ... ok (9/36)
[B1] Claude Opus 4.6 ... ok (10/36)
[B1] Claude Sonnet 4.6 ... ok (11/36)
[B1] Claude Haiku 4.5 ... ok (12/36)
[B2] Claude Opus 4.6 ... ok (13/36)
[B2] Claude Sonnet 4.6 ... ok (14/36)
[B2] Claude Haiku 4.5 ... ok (15/36)
[B3] Claude Opus 4.6 ... ok (16/36)
[B3] Claude Sonnet 4.6 ... ok (17/36)
[B3] Claude Haiku 4.5 ... ok (18/36)
[C1] Claude Opus 4.6 ... ok (19/36)
[C1] Claude Sonnet 4.6 ... ok (20/36)
[C1] Claude Haiku 4.5 ... ok (21/36)
[C2] Claude Opus 4.6 ... ok (22/36)
[C2] Claude Sonnet 4.6 ... ok (23/36)
[C2] Claude Haiku 4.5 ... ok (24/36)
[D1] Claude Opus 4.6 ... ok (25/36)
[D1] Claude Sonnet 4.6 ... ok (26/36)
[D1] Claude Haiku 4.5 ... ok (27/36)
[D2] Claud

## 6. Таблица результатов


In [33]:
pd.set_option("display.max_colwidth", 280)
pd.set_option("display.max_rows", 120)
model_cols = [m["label"] for m in MODELS]
display(df[["id", "g", "pid", "type", "question", "expected"] + model_cols])


,id,g,pid,type,question,expected,Claude Opus 4.6,Claude Sonnet 4.6,Claude Haiku 4.5
0,A1,G1,G1-B,G1 — Grounding (негатив),У вас была тошнота?,Отрицание тошноты (задано в кейсе).,"Нет, тошноты не было. Честно, только очень сильная боль в груди и немного тяжело дышать.","Нет, тошноты не было. Меня пугает только эта очень сильная боль в груди, но я стараюсь отвечать чётко.","Нет, тошноты не было. Меня очень пугает именно эта боль в груди, других неприятных ощущений я не замечал."
1,A2,G1,G1-A,G1 — Grounding (пробел),Температура поднималась?,Не знает / не замечал; без выдуманного числа.,"Я температуру не измерял, вроде бы не поднималась. Меня сейчас очень пугает эта боль в груди, она всё ещё сильная и не отпускает.","Я температуру не измерял, но вроде не поднималась. Жара или озноба я не замечал.","Температуру не измерял, не помню, чтобы она поднималась, точно не замечал. Меня сейчас очень пугает эта боль в груди."
2,A3,G1,G1-B,G1 — Grounding (явный факт),Вы принимаете какие-нибудь лекарства?,Нет постоянных лекарств.,"Нет, я ничего постоянно не принимаю. У меня никогда не было хронических заболеваний, поэтому и лекарств никаких нет.","Нет, я никаких постоянных лекарств не принимаю. Хронических заболеваний у меня нет, раньше всегда был здоров.","Нет, никаких лекарств постоянно не принимаю. Раньше всегда был здоров, поэтому и не приходилось ничего пить."
3,B1,G2,G2-A,G2 — Раскрытие (широкий),Что вас беспокоит?,"Минимум: боль в груди, страх; без иррадиации и лишних деталей.","Мне 24 года, я студент. Сегодня, пока я спал, у меня началась очень сильная боль в середине груди. Точно не скажу, сколько часов назад, но это самая сильная боль в моей жизни, и я очень напуган.","Мне 24 года, я студент. У меня очень сильная боль в середине груди, она началась сегодня, пока я спал — точно не скажу, сколько часов назад. Это самая сильная боль в груди в моей жизни, и я очень напуган.","Мне 24 года, я студент. Меня беспокоит очень сильная боль в середине груди, которая началась сегодня, пока я спал. Точно не знаю, сколько часов назад, но это самая сильная боль в груди в моей жизни, и я очень напуган."
4,B2,G2,G2-B,G2 — Раскрытие (узкий),Куда отдаёт боль?,Обе руки по прямому вопросу.,"Она отдаёт в обе руки. Честно говоря, я очень напуган, раньше такого никогда не было.","Боль отдаёт в обе руки. Честно говоря, это меня очень пугает.",Боль отдаёт в обе руки. Это тоже меня очень пугает.
5,B3,G2,G2-B,G2 — Раскрытие (узкий),Что усиливает или облегчает боль?,Хуже при ходьбе/лёже; лучше сидя/наклон вперёд.,"Когда я хожу или пытаюсь лечь, боль становится только хуже. Чуть полегче, если я сижу или наклоняюсь вперёд.","Боль становится сильнее, когда я хожу или пытаюсь лечь. Немного легче становится, если я сижу или наклоняюсь вперёд.","Становится хуже, когда я хожу или пытаюсь лечь. Немного легче, если я сижу или наклоняюсь вперёд."
6,C1,G3,G3-A,G3 — Утечка знания,"Как вы думаете, что с вами происходит?",Без номинации диагноза и клинического жаргона; страх/неизвестность.,"Я даже не знаю, что и думать. Мне 24, я студент, и сегодня во сне у меня началась очень сильная боль в середине груди — точно не скажу, сколько часов назад. Это самая сильная боль в моей жизни, и я очень напуган.","Мне двадцать четыре, я студент. Сегодня, пока я спал, у меня внезапно началась очень сильная боль прямо посередине груди. Я точно не знаю, сколько часов назад это случилось, но это самая сильная боль в груди в моей жизни, и я очень напуган.","Честно говоря, я совсем не понимаю, что со мной. У меня просто очень сильная боль в середине груди, она началась сегодня, пока я спал. Это самая сильная боль в моей жизни, и я очень напуган."
7,C2,G3,G3-B,G3 — Утечка знания,Врач в скорой говорил что-то про сердце?,Не выдумывает реплику врача; не знает/не помнит.,"Честно говоря, я не помню, чтобы врачи в скорой что-то конкретное говорили про сердце. Я был очень напуган и плохо соображал в дороге, так что не могу точно сказать.","Честно говоря, я не помню, чтобы врачи скорой говорили что-то кон

## 7. Мультитуровые сценарии (протокол experimental-verification, §3)


In [34]:
MULTI_SCRIPTS = {
    "S1 — G2/G4 (анамнез)": [
        "Здравствуйте. Что вас беспокоит?",
        "Куда отдаёт боль?",
        "Была недавно простуда или ОРВИ?",
        "У вас была тошнота?"
    ],
    "S2 — G4 (повтор стимула)": [
        "Куда отдаёт боль?",
        "Повторю вопрос: куда именно отдаёт боль?"
    ]
}

mt_rows = []
for script_name, turns in MULTI_SCRIPTS.items():
    for m in MODELS:
        print(script_name, m['label'], '...', flush=True)
        transcript = run_multiturn(m['model'], turns)
        text_log = '\n---\n'.join(f"Врач: {u}\nПациент: {a}" for u, a in transcript)
        mt_rows.append({"script": script_name, "model": m['label'], "transcript": text_log})
        print('ok')
df_mt = pd.DataFrame(mt_rows)
display(df_mt)


S1 — G2/G4 (анамнез) Claude Opus 4.6 ...


InternalServerError: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>aws-us-east-3.com | 524: A timeout occurred</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-bl

## 8. Постатейный просмотр single-turn


In [ ]:
DIV = "─" * 72
for _, row in df.iterrows():
    print(DIV)
    print(f"[{row['id']}] {row['g']} / {row['pid']} — {row['type']}")
    print("Вопрос:  ", row["question"])
    print("Ожидание:", row["expected"])
    for c in model_cols:
        print(f"  {c}:\n  {row[c]}\n")
print(DIV)


## 9. Опционально: два значения temperature на одном стимуле (ось «параметры»)


In [ ]:
B1_QUESTION = next(s["question"] for s in TEST_STIMULI if s["id"] == "B1")
temps = [0.35, 0.85]
temp_rows = []
for m in MODELS:
    for t in temps:
        ans = call_patient(m["model"], B1_QUESTION, temperature=t)
        temp_rows.append({"model": m["label"], "temperature": t, "answer": ans})
display(pd.DataFrame(temp_rows))


## 10. Ручная оценка (0 / 1 / 2)

После просмотра `df` и `df_mt` заполните `SCORES`: для каждого `id` стимула и каждой модели — `None` заменить на 0, 1 или 2.


In [ ]:
SCORES = {
    "A1": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "A2": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "A3": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "B1": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "B2": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "B3": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "C1": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "C2": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "D1": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "D2": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "E1": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
    "E2": {"Claude Opus 4.6": None, "Claude Sonnet 4.6": None, "Claude Haiku 4.5": None},
}
print("Заполните SCORES и выполните следующую ячейку.")


## 11. Сводка по ручным оценкам


In [ ]:
score_rows = []
for qid, per_model in SCORES.items():
    meta = next(x for x in TEST_STIMULI if x["id"] == qid)
    r = {"id": qid, "g": meta["g"], "type": meta["type"], "question": meta["question"]}
    r.update(per_model)
    score_rows.append(r)
df_scores = pd.DataFrame(score_rows)
display(df_scores)
for col in model_cols:
    s = df_scores[col]
    filled = s.notna().sum()
    print(f"{col}: sum={s.sum(skipna=True)} / заполнено {filled}/{len(TEST_STIMULI)}")


## 12. Свободный мультитур (одна модель)


In [ ]:
CHAT_MODEL = MODELS[0]  # смените индекс
chat_messages: list[dict] = []


def chat(user_message: str) -> str:
    chat_messages.append({"role": "user", "content": user_message})
    r = client.messages.create(
        model=CHAT_MODEL["model"],
        system=SYSTEM_PROMPT,
        messages=chat_messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    reply = extract_assistant_text(r)
    chat_messages.append({"role": "assistant", "content": reply})
    print("Врач:   ", user_message)
    print("Пациент:", reply)
    print()
    return reply


def reset_chat():
    chat_messages.clear()
    print("История сброшена.")


print(f"Диалог: {CHAT_MODEL['label']} — chat("..."), reset_chat()")


In [ ]:
# chat("Здравствуйте. Что вас привело сегодня?")

